[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/crystal_diffraction/crystal_diffraction_spacetime.ipynb)

# Crystal Deformation and Diffraction in Spacetime

This example computes how a crystal's diffraction pattern moves when the crystal deforms, with the light it scatters as an electromagnetic field and the time of an observer at rest with the crystal explicit. Stretched, a crystal's atoms move apart while its diffraction peaks move together: a peak sits where the waves from every atom arrive in phase, and when the atoms move, the peaks move so that every phase is kept. In circularly polarized light the field turns as the wave advances, its electric part into its magnetic part and back, and the phase of the wave is how far it has turned; the brightness of the pattern is the energy density of the scattered field that the observer measures.

Events are antivectors of spacetime and wavevectors vectors, a wavevector read by crossing it: the phase of a wave at an event is their pairing, `wavevector & event`. The deformation is a map on events that leaves the observer's time alone, and its adjugate carries the wavevectors across it, `deformation.adjugate()(k) & x == k & deformation(x)`, by incidence alone: no metric enters. The incident light is the null bivector `wavevector ^ mv.x`, and the pseudoscalar `I = mv.xyzt`, which squares to minus one, turns it into the same wave polarized along y, `I * incident == mv.y ^ wavevector`. Each atom's wave is the incident field turned by its phase, `(I * phases).exp() * incident`, and the intensity is the energy density: the field's sandwich of the observer's direction in time is the energy current it carries, which the observer's slice of space reads, `0.5 * (mv.t & (field >> time))`, positive for every field.

In the Riemann–Silberstein notation the field reads as the complex vector $\mathbf F = \mathbf E + i\mathbf B$, the turn by $I$ as multiplication by $i$, a circularly polarized wave as $\mathbf F_0\,e^{i\varphi}$, and the energy density as $\tfrac12\,\mathbf F^*\!\cdot\mathbf F = \tfrac12\left(\mathbf E^2 + \mathbf B^2\right)$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import STA
from examples.animation import save_figure
from examples.optics.crystal_diffraction import render

# The algebra of spacetime, with space positive, `x+y+z+t-`.
ga = STA
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
# Events are antivectors with intrinsic orientation.
Antivector = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
# The pseudoscalar squares to minus one; multiplying a field by it turns its electric part into its
# magnetic part.
I = mv.xyzt                                                                         # [] Pseudoscalar
# The observer at rest with the crystal: its slice of space `mv.t`, and its direction in time, which pairs
# to one with that slice; the dual carries the minus of t's square.
time = -mv.t.dual()                                                                 # [] Antivector

## 1. The lattice, the deformation and its adjugate

The atoms form a square layer, events at the observer's time zero. Its reciprocal lattice is the set of wavevectors whose phase is a whole number of turns at every atom, and these are where the diffraction peaks appear. The deformation is a map on events, `Antivector <- Antivector`: on space the identity plus a stretch along x and a shear, turned in the plane xy, and on the observer's time the identity. A wavevector is carried across it by the adjugate, the map with `deformation.adjugate()(k) & x == k & deformation(x)`: it pulls a phase measurement back through the deformation, using only incidence. The deformed peaks are the wavevectors `k'` with `deformation.adjugate()(k') == k`, found by one solve through the adjugate.

In matrix notation the deformed peaks read as $F^{-\mathsf T}\mathbf g$ for a deformation $F$, with $(F^{-\mathsf T}\mathbf g)^{\mathsf T}(F\mathbf x) = \mathbf g^{\mathsf T}\mathbf x$; written with the Minkowski metric $\eta$ to turn wavevectors into events and back, as the adjoint $\eta^{-1}F^{-\mathsf T}\eta$, they come out the same.

In [ ]:
# A square layer of equal atoms; one deformation per case: none, a stretch, and a stretch, a shear and a turn.
n_shells = 3
n_orders = 3
spacing = 1.0
stretches = np.array([1.0, 1.7, 1.7])
shears = np.array([0.0, 0.0, 0.6])
angles = np.array([0.0, 0.0, np.pi / 9])

In [ ]:
# The square layer's atoms, events at the observer's time zero, and its reciprocal lattice: the wavevectors
# whose phase at every atom is a whole number of turns.
cells = np.arange(-n_shells, n_shells + 1)
atoms = (mv.x * cells[None, :] + mv.y * cells[:, None]).reshape(-1).dual() * spacing   # [atoms] Antivector
reflections = np.arange(-n_orders, n_orders + 1)
lattice = ((mv.x * reflections[None, :] + mv.y * reflections[:, None]).reshape(-1)
           * (2 * np.pi / spacing))                                                 # [reflections] Vector

# An event's distance from the plane x, `mv.x & Antivector`, read along x and stretched, and its distance
# from the plane y slid along x. The observer's time lies in both planes and is left where it is.
local = (Antivector + (stretches - 1) * mv.x.dual() * (mv.x & Antivector)
         + shears * mv.x.dual() * (mv.y & Antivector))                              # [cases] Antivector <- Antivector
# A positive angle turns x toward y.
deformation = (mv.xy * (-angles / 2)).exp() >> local                                # [cases] Antivector <- Antivector
positions = deformation[:, None](atoms)                                             # [cases, atoms] Antivector

# The adjugate pulls a phase measurement back; solve for the measurement after deformation.
adjugate = deformation.adjugate()                                                   # [cases] Vector <- Vector
reciprocal = adjugate[:, None].solve(lattice)                                       # [cases, reflections] Vector

## 2. The scattered field and its energy density

The incident light is a plane wave running along z, the null bivector `wavevector ^ mv.x` with `wavevector = mv.t - mv.z`, whose phase stays put along `time + mv.z.dual()`: its electric field lies along x, its magnetic field along y, and it squares to zero. Each atom scatters it with the phase of its event, the pairing of the momentum transfer with the event, `transfers & positions`. The incident field turned by that phase, `(I * phases).exp() * incident`, is the same wave with its polarization turned about z by the phase: circularly polarized light, whose helicity is the sign of `I` in the exponent. A turn by `I` keeps a field's energy, so at every transfer the energy density the observer measures is the incident energy, one, times the squared length of the summed phase factors.

In [ ]:
# A square section of momentum transfers in the plane of the layer, one per pixel.
half_width = 10.0
n_pixels = 161
samples = np.linspace(-half_width, half_width, n_pixels)
transfers = mv.x * samples[None, :] + mv.y * samples[:, None]                       # [rows, columns] Vector
# Light running along z: its phase stays put along `time + mv.z.dual()`. Its electric field lies along x.
wavevector = mv.t - mv.z                                                            # [] Vector
incident = wavevector ^ mv.x                                                        # [] Bivector

In [ ]:
def scattered(positions: Antivector, transfers: Vector, incident: Bivector) -> Bivector:
    """The wave each atom scatters to each transfer: the incident field turned by the atom's phase."""
    # Light scattered without a change of frequency has a transfer with no part along the observer's
    # time, so the time of an atom's event does not enter its phase.
    phases = transfers[..., None] & positions[..., None, None, :]                   # [..., rows, columns, atoms] Scalar
    # I squares to minus one: its exponential turns the field by the phase, electric into magnetic.
    return (I * phases).exp() * incident                                            # [..., rows, columns, atoms] Bivector


waves = scattered(positions, transfers, incident)                                   # [cases, rows, columns, atoms] Bivector
# Equal atoms scatter equally; divided by their count, an in-phase peak carries the incident field.
field = waves.sum(axis=-1) / positions.shape[-1]                                    # [cases, rows, columns] Bivector
# The field's sandwich of the observer's direction in time is twice the energy and momentum it carries
# for that observer, and the observer's slice of space reads twice the energy density from it.
energy_density = 0.5 * (mv.t & (field >> time))                                     # [cases, rows, columns] Scalar
save_figure(render.spacetime_comparison(positions, reciprocal, transfers, energy_density), "crystal_diffraction_spacetime");

The top row shows the atoms of each case: the square layer, the stretched layer, and the stretched, sheared and turned layer. The bottom row shows the energy density of the summed field over the section of transfers, on a logarithmic scale, with cyan rings at the reciprocal lattice points carried by the adjugate.

In [ ]:
# checks
from examples.optics.crystal_diffraction import core

# Every wavevector meets every moved atom in the phase it met the atom at rest.
moved = reciprocal[:, :, None] & positions[:, None, :]                              # [cases, reflections, atoms] Scalar
np.testing.assert_allclose((moved - (lattice[:, None] & atoms[None, :])).kernel, 0.0, atol=1e-10)
# The incident wave is null, and I turns its polarization from x to y.
np.testing.assert_allclose(incident.squared().kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((I * incident - (mv.y ^ wavevector)).kernel, 0.0, atol=1e-12)
# At every moved reciprocal lattice point the atoms' waves arrive in phase, with the incident energy.
peaks = scattered(positions, reciprocal[:, None, :], incident).sum(axis=-1) / positions.shape[-1]
np.testing.assert_allclose((0.5 * (mv.t & (peaks >> time))).to_array(), 1.0, atol=1e-5)
# The energy density is non-negative, and it is the intensity computed in the algebra of space.
crystals_in_space = core.square(n_shells, n_orders, spacing).carried(core.deformation(stretches, shears, angles))
intensity_in_space = crystals_in_space.intensity(core.transfer_grid(half_width, n_pixels))
assert energy_density.to_array().min() >= 0.0
np.testing.assert_allclose(energy_density.to_array(), intensity_in_space.to_array(), atol=1e-6)